# Modeling - KNN Regression

First neighbor-based baseline for TVT prediction. This notebook tests whether local similarity in simple trajectory/GR features is useful before moving to SVR, radius neighbors, clustered KNN, XGBoost, or LightGBM.

## 1. Setup

Load packages, paths, constants, and train wells. KNN requires scaled features because distance magnitude directly controls neighbor selection.

In [ ]:
"Imports, paths, constants, and train wells."

import pickle
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for parent in ROOT.parents:
        if (parent / "src").is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import metrics_summary
from src.evaluation.validation import group_well_folds, target_zone_mask
from src.io.loaders import discover_wells, load_horizontal_well
from src.modeling.linear import Standardizer
from src.modeling.neighbors import make_neighbor_model
from src.preprocessing.tabular import make_simple_feature_frame, sample_training_rows
from src.utils.run_bundle import RunBundle
from src.utils.seeds import set_global_seeds

set_global_seeds(42)

MODEL_FAMILY = "neighbors"
MODEL_NAME = "knn_regression"
MODEL_LABEL = "KNN Regression"
N_SPLITS = 3
MAX_TRAIN_ROWS_PER_FOLD = 5_000
MAX_VALID_ROWS_PER_FOLD = 30_000
MAX_FINAL_TRAIN_ROWS = 60_000
CANDIDATES = [
    {"name": "knn_k5_distance", "params": {"n_neighbors": 5, "weights": "distance", "algorithm": "auto", "n_jobs": -1}},
    {"name": "knn_k15_distance", "params": {"n_neighbors": 15, "weights": "distance", "algorithm": "auto", "n_jobs": -1}},
    {"name": "knn_k31_distance", "params": {"n_neighbors": 31, "weights": "distance", "algorithm": "auto", "n_jobs": -1}},
]

train_dir = ROOT / "data" / "raw" / "train"
artifacts_dir = ROOT / "artifacts"
tables_dir = artifacts_dir / "tables"
figures_dir = artifacts_dir / "figures"
models_dir = artifacts_dir / "models" / "neighbors"
for directory in [tables_dir, figures_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)

wells = discover_wells(train_dir)
print("model:", MODEL_LABEL)
print("train wells:", len(wells))
print("folds:", N_SPLITS)
print("train cap per fold:", MAX_TRAIN_ROWS_PER_FOLD)
print("validation cap per fold:", MAX_VALID_ROWS_PER_FOLD)

## 2. Feature Matrix Helpers

Build simple test-safe features, split known rows from target-zone rows, and use deterministic row caps for KNN runtime control.

In [ ]:
"Feature extraction, deterministic validation sampling, and comparison logging."


def load_fold_matrix(well_names, zone):
    frames = []
    targets = []
    row_counts = []
    for well in well_names:
        df = load_horizontal_well(train_dir / f"{well}__horizontal_well.csv")
        if zone == "known":
            mask = df["TVT_input"].notna() & df["TVT"].notna()
        elif zone == "target":
            mask = target_zone_mask(df)
        else:
            raise ValueError("zone must be known or target")
        feat = make_simple_feature_frame(df).loc[mask]
        frames.append(feat)
        targets.append(df.loc[mask, "TVT"].to_numpy(dtype=float))
        row_counts.append(int(mask.sum()))
    X_df = pd.concat(frames, axis=0)
    y = np.concatenate(targets)
    return X_df, y, row_counts


def sample_validation_rows(X, y, max_rows):
    if len(y) <= max_rows:
        return X, y
    idx = np.linspace(0, len(y) - 1, max_rows).astype(int)
    return X.iloc[idx].reset_index(drop=True), y[idx]


def append_model_comparison(row):
    path = tables_dir / "model_comparison.csv"
    new_row = pd.DataFrame([row])
    if path.exists():
        table = pd.read_csv(path)
        table = pd.concat([table, new_row], ignore_index=True)
    else:
        table = new_row
    table = table.drop_duplicates(["model_family", "model_name"], keep="last")
    table.to_csv(path, index=False)
    return path

folds = group_well_folds(wells, n_splits=N_SPLITS)
print("fold sizes:", [(len(train), len(valid)) for train, valid in folds])

## 3. KNN Sweep

For each candidate, scale features using the training fold only, fit KNN on capped known-zone rows, and score a deterministic target-zone validation sample.

In [ ]:
"Run grouped-well CV for a small KNN sweep."

start = time.time()
records = []
fold_predictions = []

for candidate in CANDIDATES:
    candidate_records = []
    candidate_predictions = []
    for fold_idx, (train_wells, valid_wells) in enumerate(folds, start=1):
        X_train_df, y_train, train_counts = load_fold_matrix(train_wells, zone="known")
        X_valid_df, y_valid, valid_counts = load_fold_matrix(valid_wells, zone="target")
        X_valid_df, y_valid = sample_validation_rows(X_valid_df, y_valid, MAX_VALID_ROWS_PER_FOLD)

        feature_columns = list(X_train_df.columns)
        feature_medians = X_train_df.median(numeric_only=True)
        X_train = X_train_df.fillna(feature_medians).to_numpy(dtype=float)
        X_valid = X_valid_df.reindex(columns=feature_columns).fillna(feature_medians).to_numpy(dtype=float)
        X_train, y_train = sample_training_rows(X_train, y_train, MAX_TRAIN_ROWS_PER_FOLD)

        scaler = Standardizer()
        X_train_scaled = scaler.fit_transform(X_train)
        X_valid_scaled = scaler.transform(X_valid)

        model = make_neighbor_model("knn", candidate["params"])
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_valid_scaled)
        summary = metrics_summary(y_valid, pred)

        record = {
            "candidate": candidate["name"],
            "fold": fold_idx,
            "rmse": summary["rmse"],
            "mae": summary["mae"],
            "r2": summary["r2"],
            "train_rows_used": len(y_train),
            "valid_rows_scored": len(y_valid),
            "valid_rows_available": int(sum(valid_counts)),
            "n_features": len(feature_columns),
        }
        candidate_records.append(record)
        candidate_predictions.append(pd.DataFrame({
            "candidate": candidate["name"],
            "fold": fold_idx,
            "y_true": y_valid,
            "y_pred": pred,
            "residual": y_valid - pred,
        }))
        print(f"{candidate['name']} fold {fold_idx}: rmse={summary['rmse']:.4f}, scored_rows={len(y_valid)}")

    records.extend(candidate_records)
    fold_predictions.extend(candidate_predictions)

metrics_df = pd.DataFrame(records)
pred_df = pd.concat(fold_predictions, ignore_index=True)
summary_df = (
    metrics_df.groupby("candidate", as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        train_rows_used=("train_rows_used", "mean"),
        valid_rows_scored=("valid_rows_scored", "mean"),
        n_features=("n_features", "max"),
    )
    .sort_values("rmse_mean")
)
best_row = summary_df.iloc[0]
best_candidate = next(item for item in CANDIDATES if item["name"] == best_row["candidate"])
elapsed = time.time() - start

summary_df

## 4. Visual Diagnostics

Plot candidate RMSE and residuals for the best KNN candidate.

In [ ]:
"Create RMSE and residual diagnostic figures."

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary_df["candidate"], summary_df["rmse_mean"], yerr=summary_df["rmse_std"].fillna(0.0), color="#3a7ca5")
ax.set_ylabel("CV RMSE")
ax.set_title("KNN Candidate RMSE")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

best_pred = pred_df[pred_df["candidate"] == best_row["candidate"]]
fig2, ax2 = plt.subplots(figsize=(7, 4))
sample_plot = best_pred.iloc[np.linspace(0, len(best_pred) - 1, min(5000, len(best_pred))).astype(int)]
ax2.scatter(sample_plot["y_true"], sample_plot["residual"], s=3, alpha=0.25, color="#d1495b")
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xlabel("True TVT")
ax2.set_ylabel("Residual")
ax2.set_title(f"Residuals - {best_row['candidate']}")
fig2.tight_layout()

## 5. Final Fit And Run Bundle

Fit the selected KNN on a larger deterministic known-zone sample, save reportable artifacts, and update the comparison table.

In [ ]:
"Fit final KNN model, save artifacts, and write run metadata."

X_all_df, y_all, all_counts = load_fold_matrix(wells, zone="known")
feature_columns = list(X_all_df.columns)
feature_medians = X_all_df.median(numeric_only=True)
X_all = X_all_df.fillna(feature_medians).to_numpy(dtype=float)
X_all, y_all = sample_training_rows(X_all, y_all, MAX_FINAL_TRAIN_ROWS)

final_scaler = Standardizer()
X_all_scaled = final_scaler.fit_transform(X_all)
final_model = make_neighbor_model("knn", best_candidate["params"])
final_model.fit(X_all_scaled, y_all)

config = {
    "model": MODEL_NAME,
    "best_candidate": best_row["candidate"],
    "candidate_params": best_candidate["params"],
    "n_splits": N_SPLITS,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "max_valid_rows_per_fold": MAX_VALID_ROWS_PER_FOLD,
    "max_final_train_rows": MAX_FINAL_TRAIN_ROWS,
    "feature_columns": feature_columns,
    "validation_note": "Deterministic sampled target-zone validation for KNN runtime control.",
}

with RunBundle(f"neighbor_{MODEL_NAME}", config) as run:
    metrics_path = tables_dir / f"model__{MODEL_NAME}_metrics__{run.run_id}.csv"
    summary_path = tables_dir / f"model__{MODEL_NAME}_summary__{run.run_id}.csv"
    rmse_path = figures_dir / f"model__{MODEL_NAME}_rmse_comparison__{run.run_id}.png"
    residual_path = figures_dir / f"model__{MODEL_NAME}_residuals__{run.run_id}.png"
    model_path = models_dir / f"model__{MODEL_NAME}__{run.run_id}.pkl"

    metrics_df.to_csv(metrics_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    fig.savefig(rmse_path, dpi=160)
    fig2.savefig(residual_path, dpi=160)
    with open(model_path, "wb") as handle:
        pickle.dump(
            {
                "model": final_model,
                "scaler": final_scaler,
                "feature_columns": feature_columns,
                "feature_medians": feature_medians,
            },
            handle,
        )

    for metric_name, metric_value in {
        "cv_rmse_mean": float(best_row["rmse_mean"]),
        "cv_rmse_std": float(best_row["rmse_std"]),
        "cv_mae_mean": float(best_row["mae_mean"]),
        "cv_r2_mean": float(best_row["r2_mean"]),
        "train_time_s": float(elapsed),
        "n_features": int(best_row["n_features"]),
        "valid_rows_scored_per_fold": int(best_row["valid_rows_scored"]),
    }.items():
        run.add_metric(metric_name, metric_value)
    for output_path in [metrics_path, summary_path, rmse_path, residual_path, model_path]:
        run.add_output(str(output_path))
    run.set_notes(f"Best KNN candidate: {best_row['candidate']} with sampled CV RMSE {best_row['rmse_mean']:.4f}.")

comparison_path = append_model_comparison({
    "run_id": run.run_id,
    "model_family": MODEL_FAMILY,
    "model_name": str(best_row["candidate"]),
    "cv_rmse_mean": float(best_row["rmse_mean"]),
    "cv_rmse_std": float(best_row["rmse_std"]),
    "train_time_s": float(elapsed),
    "n_features": int(best_row["n_features"]),
    "notes": "KNN distance-weighted sweep; sampled target-zone validation for runtime control",
})

print("run_id:", run.run_id)
print("best:", best_row.to_dict())
print("metrics:", metrics_path)
print("summary:", summary_path)
print("figures:", rmse_path, residual_path)
print("comparison:", comparison_path)